# 096 — Proyecto: pipeline creativo trazable

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución explicada

**Ejercicio 1.** 102 + 97 + 114 + 111 = 424; 424 mod 256 = 168 = **0xA8**.
El hash didáctico solo detecta cambios por accidente estadístico (8 bits); uno real
(SHA-256) hace la colisión deliberada computacionalmente inviable.

**Ejercicio 2.** La cadena del ejemplo cierra: h(s1) = 0xF4, h(s2) = 0xE6,
h(s3) = 0x05, y cada `input_hash` coincide con el `output_hash` anterior
(F4 → E6 → 05). El verificador solo necesita las salidas y los manifiestos: no
necesita re-ejecutar los modelos.

**Ejercicio 3.** Con seed=8, h(s2') = 0xE7. El manifiesto de la etapa 3 declara
`input_hash = 0xE6`, así que la verificación **rompe en el eslabón 2→3**. La cadena
detecta *que* los bytes consumidos por la etapa 3 ya no son los declarados; NO detecta
qué campo cambió, quién lo cambió ni por qué — eso exige el registro append-only y,
en producción, firmas (C2PA) que añaden no repudio sobre la integridad.

**Ejercicio 4.** El contrato JSON expone `kind` y `evidence`; la evidencia es lo único
que autoriza conclusiones.

In [ ]:
result = run_lab("capstone", seed=96)
assert result["kind"] == "capstone"
assert result["evidence"]
show(result)


In [ ]:
# Verificación numérica de los ejercicios
def toy_hash(s):
    return sum(ord(c) for c in s) % 256

# Ejercicio 1
print(f"h('faro') = {hex(toy_hash('faro'))}")
assert toy_hash("faro") == 0xA8

# Ejercicio 2: pipeline de 3 etapas con hashes encadenados
salidas = [
    "un faro al amanecer",
    "IMG[faro,amanecer,seed=7]",
    "IMG[faro,amanecer,seed=7]+recorte",
]
manifiestos = []
prev = None
for i, s in enumerate(salidas, start=1):
    m = {"stage": i, "input_hash": prev, "output_hash": toy_hash(s)}
    manifiestos.append(m)
    prev = m["output_hash"]

def verificar(manifiestos, salidas):
    prev = None
    for m, s in zip(manifiestos, salidas):
        if m["input_hash"] != prev:
            return f"CADENA ROTA en la etapa {m['stage']} (input_hash no coincide)"
        if toy_hash(s) != m["output_hash"]:
            return f"CADENA ROTA en la etapa {m['stage']} (contenido alterado)"
        prev = m["output_hash"]
    return "VERIFICA: " + " → ".join(f"{m['output_hash']:02X}" for m in manifiestos)

print(verificar(manifiestos, salidas))
assert [m["output_hash"] for m in manifiestos] == [0xF4, 0xE6, 0x05]

# Ejercicio 3: alterar la etapa 2 sin actualizar los manifiestos
salidas_alteradas = list(salidas)
salidas_alteradas[1] = "IMG[faro,amanecer,seed=8]"
print(f"h(s2 alterada) = {hex(toy_hash(salidas_alteradas[1]))} (esperado 0xE6)")
print(verificar(manifiestos, salidas_alteradas))
assert "ROTA" in verificar(manifiestos, salidas_alteradas)

## Reflexión

1. Con la misma semilla pero una versión nueva del modelo en la etapa 2, la salida cambia y la cadena de hashes rompe. ¿Qué campo del contrato explica la diferencia y por qué semilla sin versión no basta para auditar?
2. ¿Por qué un manifiesto reconstruido después de la publicación no constituye evidencia de procedencia aunque todos los hashes cuadren?
3. ¿Qué añade una firma criptográfica (C2PA) sobre la cadena de hashes simple de este proyecto, y qué problema operativo nuevo introduce?